# Fundamentals of Quantitative Finance Using Python

This notebook covers the core concepts and techniques in quantitative finance, including:
- Time value of money
- Return calculations and risk metrics
- Portfolio theory
- Bond pricing and yield curves
- Option pricing basics
- Monte Carlo simulation

We'll use Python libraries such as NumPy, pandas, and matplotlib for financial computations and visualization.

## Learning ObjectivesBy the end of this notebook, you will be able to:- Understand time value of money and discounting principles- Calculate returns and apply portfolio theory fundamentals- Price bonds using present value techniques- Implement the Black-Scholes option pricing model- Apply Monte Carlo simulation for derivative pricing**Estimated Time:** 90-120 minutes---

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize
import seaborn as sns

# Set style for better-looking plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set random seed for reproducibility
np.random.seed(42)

## 1. Time Value of Money

The time value of money is a fundamental concept in finance: a dollar today is worth more than a dollar tomorrow.

### Present Value (PV) and Future Value (FV)

$$FV = PV \times (1 + r)^n$$
$$PV = \frac{FV}{(1 + r)^n}$$

Where:
- $r$ = discount rate
- $n$ = number of periods

In [ ]:
def present_value(future_value, rate, periods):
    """Calculate present value of a future cash flow."""
    return future_value / (1 + rate) ** periods

def future_value(present_value, rate, periods):
    """Calculate future value of a present cash flow."""
    return present_value * (1 + rate) ** periods

# Example: What is $1000 in 5 years worth today at 5% discount rate?
pv = present_value(1000, 0.05, 5)
print(f"Present Value: ${pv:.2f}")

# Example: What will $1000 be worth in 5 years at 5% interest?
fv = future_value(1000, 0.05, 5)
print(f"Future Value: ${fv:.2f}")

### Net Present Value (NPV)

NPV is used to evaluate investment opportunities by discounting all future cash flows to present value.

$$NPV = \sum_{t=0}^{n} \frac{CF_t}{(1 + r)^t}$$

In [ ]:
def npv(cash_flows, discount_rate):
    """Calculate Net Present Value of a series of cash flows."""
    return sum([cf / (1 + discount_rate) ** t for t, cf in enumerate(cash_flows)])

# Example: Investment of -$10,000 with returns over 5 years
cash_flows = [-10000, 3000, 3500, 4000, 4500, 5000]
discount_rate = 0.08

npv_value = npv(cash_flows, discount_rate)
print(f"NPV: ${npv_value:.2f}")
print(f"Investment is {'profitable' if npv_value > 0 else 'not profitable'}")

## 2. Returns and Risk Metrics

### Simple and Logarithmic Returns

**Simple Return:**
$$R_t = \frac{P_t - P_{t-1}}{P_{t-1}}$$

**Logarithmic Return:**
$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$$

In [ ]:
# Generate sample price data
dates = pd.date_range('2023-01-01', periods=252, freq='D')
prices = 100 * np.exp(np.cumsum(np.random.normal(0.0005, 0.02, 252)))

df = pd.DataFrame({'Price': prices}, index=dates)

# Calculate returns
df['Simple_Return'] = df['Price'].pct_change()
df['Log_Return'] = np.log(df['Price'] / df['Price'].shift(1))

# Display statistics
print("Return Statistics:")
print(df[['Simple_Return', 'Log_Return']].describe())

# Plot price and returns
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

df['Price'].plot(ax=axes[0], title='Asset Price', color='blue')
axes[0].set_ylabel('Price ($)')

df['Log_Return'].plot(ax=axes[1], title='Log Returns', color='green', alpha=0.7)
axes[1].set_ylabel('Return')
axes[1].axhline(y=0, color='r', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### Volatility (Standard Deviation)

Volatility measures the dispersion of returns and is a key risk metric.

$$\sigma = \sqrt{\frac{1}{N-1} \sum_{i=1}^{N} (r_i - \bar{r})^2}$$

In [ ]:
# Calculate volatility
daily_volatility = df['Log_Return'].std()
annual_volatility = daily_volatility * np.sqrt(252)  # Annualized

print(f"Daily Volatility: {daily_volatility:.4f} ({daily_volatility*100:.2f}%)")
print(f"Annualized Volatility: {annual_volatility:.4f} ({annual_volatility*100:.2f}%)")

# Rolling volatility
df['Rolling_Vol'] = df['Log_Return'].rolling(window=20).std() * np.sqrt(252)

plt.figure(figsize=(12, 6))
df['Rolling_Vol'].plot(title='20-Day Rolling Volatility (Annualized)', color='purple')
plt.ylabel('Volatility')
plt.axhline(y=annual_volatility, color='r', linestyle='--', label='Overall Volatility')
plt.legend()
plt.show()

### Sharpe Ratio

The Sharpe ratio measures risk-adjusted returns:

$$\text{Sharpe Ratio} = \frac{\bar{r} - r_f}{\sigma}$$

Where $r_f$ is the risk-free rate.

In [ ]:
def sharpe_ratio(returns, risk_free_rate=0.02):
    """Calculate annualized Sharpe ratio."""
    excess_returns = returns - risk_free_rate/252
    return np.sqrt(252) * excess_returns.mean() / returns.std()

sr = sharpe_ratio(df['Log_Return'].dropna())
print(f"Sharpe Ratio: {sr:.4f}")

## 3. Portfolio Theory

### Modern Portfolio Theory (MPT)

Portfolio return and variance:

$$r_p = \sum_{i=1}^{n} w_i r_i$$

$$\sigma_p^2 = \sum_{i=1}^{n}\sum_{j=1}^{n} w_i w_j \sigma_{ij}$$

In [ ]:
# Generate returns for multiple assets
n_assets = 4
n_periods = 252

# Simulate correlated asset returns
mean_returns = np.array([0.08, 0.12, 0.10, 0.15]) / 252  # Daily returns
cov_matrix = np.array([
    [0.04, 0.01, 0.02, 0.015],
    [0.01, 0.06, 0.025, 0.02],
    [0.02, 0.025, 0.05, 0.03],
    [0.015, 0.02, 0.03, 0.08]
]) / 252  # Daily covariance

returns = np.random.multivariate_normal(mean_returns, cov_matrix, n_periods)
returns_df = pd.DataFrame(returns, columns=['Asset A', 'Asset B', 'Asset C', 'Asset D'])

# Calculate correlation matrix
correlation_matrix = returns_df.corr()

print("Correlation Matrix:")
print(correlation_matrix)

# Visualize correlation
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Asset Return Correlations')
plt.show()

In [ ]:
def portfolio_performance(weights, returns):
    """Calculate portfolio return and volatility."""
    portfolio_return = np.sum(returns.mean() * weights) * 252
    portfolio_std = np.sqrt(np.dot(weights.T, np.dot(returns.cov() * 252, weights)))
    return portfolio_return, portfolio_std

# Simulate random portfolios
n_portfolios = 10000
results = np.zeros((3, n_portfolios))

for i in range(n_portfolios):
    weights = np.random.random(n_assets)
    weights /= np.sum(weights)
    
    portfolio_return, portfolio_std = portfolio_performance(weights, returns_df)
    
    results[0, i] = portfolio_return
    results[1, i] = portfolio_std
    results[2, i] = (portfolio_return - 0.02) / portfolio_std  # Sharpe ratio

# Plot efficient frontier
plt.figure(figsize=(12, 8))
plt.scatter(results[1, :], results[0, :], c=results[2, :], cmap='viridis', alpha=0.5)
plt.colorbar(label='Sharpe Ratio')
plt.xlabel('Volatility (Std Dev)')
plt.ylabel('Expected Return')
plt.title('Efficient Frontier - Random Portfolios')

# Mark maximum Sharpe ratio portfolio
max_sharpe_idx = np.argmax(results[2, :])
plt.scatter(results[1, max_sharpe_idx], results[0, max_sharpe_idx], 
           marker='*', color='red', s=500, label='Max Sharpe Ratio')
plt.legend()
plt.show()

print(f"Maximum Sharpe Ratio: {results[2, max_sharpe_idx]:.4f}")
print(f"Return: {results[0, max_sharpe_idx]:.4f}")
print(f"Volatility: {results[1, max_sharpe_idx]:.4f}")

## 4. Bond Pricing

### Zero-Coupon Bond Pricing

$$P = \frac{F}{(1 + r)^n}$$

Where:
- $P$ = bond price
- $F$ = face value
- $r$ = yield
- $n$ = time to maturity

In [ ]:
def zero_coupon_bond_price(face_value, yield_rate, time_to_maturity):
    """Calculate price of a zero-coupon bond."""
    return face_value / (1 + yield_rate) ** time_to_maturity

# Example
face_value = 1000
yield_rate = 0.05
time_to_maturity = 10

price = zero_coupon_bond_price(face_value, yield_rate, time_to_maturity)
print(f"Zero-Coupon Bond Price: ${price:.2f}")

### Coupon Bond Pricing

$$P = \sum_{t=1}^{n} \frac{C}{(1 + r)^t} + \frac{F}{(1 + r)^n}$$

Where $C$ is the coupon payment.

In [ ]:
def coupon_bond_price(face_value, coupon_rate, yield_rate, periods):
    """Calculate price of a coupon-paying bond."""
    coupon = face_value * coupon_rate
    price = sum([coupon / (1 + yield_rate) ** t for t in range(1, periods + 1)])
    price += face_value / (1 + yield_rate) ** periods
    return price

# Example: 5% coupon bond, 4% yield, 10 years
bond_price = coupon_bond_price(1000, 0.05, 0.04, 10)
print(f"Coupon Bond Price: ${bond_price:.2f}")

# Visualize bond price vs yield relationship
yields = np.linspace(0.01, 0.10, 100)
prices = [coupon_bond_price(1000, 0.05, y, 10) for y in yields]

plt.figure(figsize=(10, 6))
plt.plot(yields * 100, prices, linewidth=2)
plt.xlabel('Yield (%)')
plt.ylabel('Bond Price ($)')
plt.title('Bond Price vs Yield (Inverse Relationship)')
plt.grid(True, alpha=0.3)
plt.axhline(y=1000, color='r', linestyle='--', label='Par Value')
plt.legend()
plt.show()

### Duration and Convexity

**Macaulay Duration:**
$$D = \frac{\sum_{t=1}^{n} t \cdot PV(CF_t)}{P}$$

**Modified Duration:**
$$D_{mod} = \frac{D}{1 + y}$$

In [ ]:
def bond_duration(face_value, coupon_rate, yield_rate, periods):
    """Calculate Macaulay and Modified duration."""
    coupon = face_value * coupon_rate
    
    # Calculate bond price and weighted cash flows
    price = 0
    weighted_cf = 0
    
    for t in range(1, periods + 1):
        pv_coupon = coupon / (1 + yield_rate) ** t
        price += pv_coupon
        weighted_cf += t * pv_coupon
    
    pv_principal = face_value / (1 + yield_rate) ** periods
    price += pv_principal
    weighted_cf += periods * pv_principal
    
    macaulay_duration = weighted_cf / price
    modified_duration = macaulay_duration / (1 + yield_rate)
    
    return macaulay_duration, modified_duration

mac_dur, mod_dur = bond_duration(1000, 0.05, 0.04, 10)
print(f"Macaulay Duration: {mac_dur:.4f} years")
print(f"Modified Duration: {mod_dur:.4f}")

## 5. Option Pricing - Black-Scholes Model

The Black-Scholes formula for European options:

**Call Option:**
$$C = S_0 N(d_1) - K e^{-rT} N(d_2)$$

**Put Option:**
$$P = K e^{-rT} N(-d_2) - S_0 N(-d_1)$$

Where:
$$d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}$$
$$d_2 = d_1 - \sigma\sqrt{T}$$

In [ ]:
def black_scholes(S, K, T, r, sigma, option_type='call'):
    """
    Calculate Black-Scholes option price.
    
    Parameters:
    S: Current stock price
    K: Strike price
    T: Time to maturity (years)
    r: Risk-free rate
    sigma: Volatility
    option_type: 'call' or 'put'
    """
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    if option_type == 'call':
        price = S * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.cdf(d2)
    else:
        price = K * np.exp(-r * T) * stats.norm.cdf(-d2) - S * stats.norm.cdf(-d1)
    
    return price

# Example
S = 100  # Current stock price
K = 100  # Strike price
T = 1    # 1 year to maturity
r = 0.05 # 5% risk-free rate
sigma = 0.2  # 20% volatility

call_price = black_scholes(S, K, T, r, sigma, 'call')
put_price = black_scholes(S, K, T, r, sigma, 'put')

print(f"Call Option Price: ${call_price:.2f}")
print(f"Put Option Price: ${put_price:.2f}")
print(f"Put-Call Parity Check: {call_price - put_price:.4f} vs {S - K * np.exp(-r * T):.4f}")

In [ ]:
# Visualize option prices vs stock price
stock_prices = np.linspace(50, 150, 100)
call_prices = [black_scholes(s, K, T, r, sigma, 'call') for s in stock_prices]
put_prices = [black_scholes(s, K, T, r, sigma, 'put') for s in stock_prices]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Call option
ax1.plot(stock_prices, call_prices, label='Call Option', linewidth=2)
ax1.plot(stock_prices, np.maximum(stock_prices - K, 0), 
         label='Intrinsic Value', linestyle='--', alpha=0.7)
ax1.axvline(x=K, color='r', linestyle=':', alpha=0.5, label='Strike Price')
ax1.set_xlabel('Stock Price')
ax1.set_ylabel('Option Price')
ax1.set_title('Call Option Price')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Put option
ax2.plot(stock_prices, put_prices, label='Put Option', linewidth=2, color='orange')
ax2.plot(stock_prices, np.maximum(K - stock_prices, 0), 
         label='Intrinsic Value', linestyle='--', alpha=0.7)
ax2.axvline(x=K, color='r', linestyle=':', alpha=0.5, label='Strike Price')
ax2.set_xlabel('Stock Price')
ax2.set_ylabel('Option Price')
ax2.set_title('Put Option Price')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Monte Carlo Simulation

Monte Carlo methods are widely used in quantitative finance for pricing derivatives, risk management, and portfolio optimization.

### Simulating Stock Prices (Geometric Brownian Motion)

$$dS_t = \mu S_t dt + \sigma S_t dW_t$$

Discrete approximation:
$$S_{t+\Delta t} = S_t \exp\left((\mu - \frac{\sigma^2}{2})\Delta t + \sigma \sqrt{\Delta t} Z\right)$$

Where $Z \sim N(0,1)$

In [ ]:
def simulate_gbm(S0, mu, sigma, T, n_steps, n_simulations):
    """
    Simulate stock price paths using Geometric Brownian Motion.
    
    Parameters:
    S0: Initial stock price
    mu: Drift (expected return)
    sigma: Volatility
    T: Time horizon (years)
    n_steps: Number of time steps
    n_simulations: Number of simulation paths
    """
    dt = T / n_steps
    paths = np.zeros((n_steps + 1, n_simulations))
    paths[0] = S0
    
    for t in range(1, n_steps + 1):
        Z = np.random.standard_normal(n_simulations)
        paths[t] = paths[t-1] * np.exp((mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z)
    
    return paths

# Simulate stock price paths
S0 = 100
mu = 0.10  # 10% expected return
sigma = 0.25  # 25% volatility
T = 1  # 1 year
n_steps = 252
n_simulations = 1000

paths = simulate_gbm(S0, mu, sigma, T, n_steps, n_simulations)

# Plot simulations
plt.figure(figsize=(12, 6))
time = np.linspace(0, T, n_steps + 1)

# Plot a subset of paths
for i in range(min(100, n_simulations)):
    plt.plot(time, paths[:, i], alpha=0.1, color='blue')

# Plot mean path
mean_path = paths.mean(axis=1)
plt.plot(time, mean_path, color='red', linewidth=2, label='Mean Path')

# Plot confidence intervals
percentile_5 = np.percentile(paths, 5, axis=1)
percentile_95 = np.percentile(paths, 95, axis=1)
plt.plot(time, percentile_5, '--', color='green', label='5th Percentile')
plt.plot(time, percentile_95, '--', color='green', label='95th Percentile')

plt.xlabel('Time (years)')
plt.ylabel('Stock Price')
plt.title(f'Monte Carlo Simulation: {n_simulations} Stock Price Paths')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Final Price Statistics:")
print(f"Mean: ${paths[-1].mean():.2f}")
print(f"Std Dev: ${paths[-1].std():.2f}")
print(f"5th Percentile: ${np.percentile(paths[-1], 5):.2f}")
print(f"95th Percentile: ${np.percentile(paths[-1], 95):.2f}")

### Monte Carlo Option Pricing

We can price European options using Monte Carlo simulation by:
1. Simulating many stock price paths
2. Calculating the payoff for each path
3. Taking the average and discounting to present value

In [ ]:
def monte_carlo_option_price(S0, K, T, r, sigma, n_simulations, option_type='call'):
    """
    Price European option using Monte Carlo simulation.
    """
    # Simulate final stock prices
    Z = np.random.standard_normal(n_simulations)
    ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    
    # Calculate payoffs
    if option_type == 'call':
        payoffs = np.maximum(ST - K, 0)
    else:
        payoffs = np.maximum(K - ST, 0)
    
    # Discount to present value
    option_price = np.exp(-r * T) * payoffs.mean()
    
    # Calculate standard error
    std_error = np.exp(-r * T) * payoffs.std() / np.sqrt(n_simulations)
    
    return option_price, std_error

# Price options using Monte Carlo
mc_call_price, mc_call_se = monte_carlo_option_price(S, K, T, r, sigma, 100000, 'call')
mc_put_price, mc_put_se = monte_carlo_option_price(S, K, T, r, sigma, 100000, 'put')

# Compare with Black-Scholes
bs_call = black_scholes(S, K, T, r, sigma, 'call')
bs_put = black_scholes(S, K, T, r, sigma, 'put')

print("Option Pricing Comparison:")
print(f"\nCall Option:")
print(f"  Monte Carlo: ${mc_call_price:.4f} ± ${mc_call_se:.4f}")
print(f"  Black-Scholes: ${bs_call:.4f}")
print(f"  Difference: ${abs(mc_call_price - bs_call):.4f}")

print(f"\nPut Option:")
print(f"  Monte Carlo: ${mc_put_price:.4f} ± ${mc_put_se:.4f}")
print(f"  Black-Scholes: ${bs_put:.4f}")
print(f"  Difference: ${abs(mc_put_price - bs_put):.4f}")

### Value at Risk (VaR)

VaR estimates the potential loss in portfolio value over a given time period at a specific confidence level.

In [ ]:
def calculate_var(returns, confidence_level=0.95, method='historical'):
    """
    Calculate Value at Risk.
    
    Methods:
    - historical: Use historical returns distribution
    - parametric: Assume normal distribution
    """
    if method == 'historical':
        var = np.percentile(returns, (1 - confidence_level) * 100)
    elif method == 'parametric':
        mu = returns.mean()
        sigma = returns.std()
        var = stats.norm.ppf(1 - confidence_level, mu, sigma)
    
    return var

def calculate_cvar(returns, confidence_level=0.95):
    """Calculate Conditional VaR (Expected Shortfall)."""
    var = calculate_var(returns, confidence_level)
    cvar = returns[returns <= var].mean()
    return cvar

# Generate portfolio returns
portfolio_returns = np.random.normal(0.0008, 0.02, 1000)

# Calculate VaR and CVaR at 95% confidence
var_95 = calculate_var(portfolio_returns, 0.95, 'historical')
cvar_95 = calculate_cvar(portfolio_returns, 0.95)

print(f"95% VaR (1-day): {var_95*100:.2f}%")
print(f"95% CVaR (1-day): {cvar_95*100:.2f}%")

# Visualize
plt.figure(figsize=(12, 6))
plt.hist(portfolio_returns * 100, bins=50, alpha=0.7, edgecolor='black')
plt.axvline(x=var_95*100, color='red', linestyle='--', linewidth=2, label=f'VaR 95%: {var_95*100:.2f}%')
plt.axvline(x=cvar_95*100, color='orange', linestyle='--', linewidth=2, label=f'CVaR 95%: {cvar_95*100:.2f}%')
plt.xlabel('Daily Return (%)')
plt.ylabel('Frequency')
plt.title('Portfolio Returns Distribution with VaR and CVaR')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Summary

This notebook covered fundamental concepts in quantitative finance:

1. **Time Value of Money**: Present/future value, NPV calculations
2. **Returns and Risk**: Simple and log returns, volatility, Sharpe ratio
3. **Portfolio Theory**: Diversification, efficient frontier, optimal portfolios
4. **Bond Pricing**: Zero-coupon and coupon bonds, duration, convexity
5. **Option Pricing**: Black-Scholes model for European options
6. **Monte Carlo Simulation**: Stock price simulation, option pricing, VaR

These concepts form the foundation for more advanced topics in quantitative finance, including:
- Advanced derivative pricing (exotic options, interest rate derivatives)
- Risk management and hedging strategies
- High-frequency trading and market microstructure
- Machine learning in finance
- Credit risk modeling

For further learning, explore:
- **Libraries**: QuantLib, PyMC, statsmodels, arch
- **Resources**: Hull's "Options, Futures, and Other Derivatives", Wilmott on Quantitative Finance
- **Practice**: Implement trading strategies, backtest portfolios, explore real market data

---## References### Academic Papers and Books1. Hull, J.C. (2021). *Options, Futures, and Other Derivatives (11th ed.)*. Pearson. Comprehensive derivatives and quantitative finance textbook.2. Luenberger, D.G. (1997). *Investment Science*. Oxford University Press. Foundational quantitative finance principles.3. Wilmott, P. (2006). *Paul Wilmott on Quantitative Finance*. Wiley. Three-volume comprehensive treatment.### Online Resources1. **QuantLib Documentation** - https://www.quantlib.org/ - Open-source quantitative finance library.2. **Quantitative Finance on arXiv** - https://arxiv.org/archive/q-fin - Latest research papers.3. **Financial Python** - https://github.com/quantopian - Quantopian lectures and tutorials.### Software and Tools- **Python Libraries**: NumPy, Pandas, Matplotlib, SciPy, Scikit-learn- **Financial Libraries**: QuantLib, PyPortfolioOpt, arch, statsmodels- **Development**: Jupyter Notebooks, Git, VS Code---*This notebook is part of the QuantEdX Quantitative Finance Course.**For questions or feedback, please contact: content@quantedx.com*